## Setup

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt

# Requires TensorFlow >=2.11 for the GroupNormalization layer.
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import Input, ConvLSTM2D, BatchNormalization, Conv3D
from tensorflow.keras.callbacks import *

import os
import sys

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

2026-02-02 11:18:41.742689: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-02 11:18:41.761805: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-02 11:18:41.761826: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-02 11:18:41.762282: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-02 11:18:41.765385: I tensorflow/core/platform/cpu_feature_guar

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Hyperparameters

In [3]:
batch_size = 256
num_epochs = 800         # Just for the sake of demonstration
total_timesteps = 750   # 1000
norm_groups = 8          # Number of groups used in GroupNormalization layer
learning_rate = 1e-4

img_size_H = 32
img_size_W = 64
img_channels = 5

first_conv_channels = 64
channel_multiplier = [1, 2, 4, 8]
widths = [first_conv_channels * mult for mult in channel_multiplier]
has_attention = [False, False, True, True]
num_res_blocks = 2  # Number of residual blocks

## Gaussian diffusion utilities

We define the **forward process** and the **reverse process** as a separate utility. Most of the code in this utility has been borrowed
from the original implementation with some slight modifications.

In [4]:
from layers.diffusion import GaussianDiffusion

## Network architecture

U-Net, originally developed for semantic segmentation, is an architecture that is
widely used for implementing diffusion models but with some slight modifications:

1. The network accepts two inputs: Image and time step
2. Self-attention between the convolution blocks once we reach a specific resolution
(16x16 in the paper)
3. Group Normalization instead of weight normalization

We implement most of the things as used in the original paper. We use the
`swish` activation function throughout the network. We use the variance scaling
kernel initializer.

The only difference here is the number of groups used for the
`GroupNormalization` layer. For the flowers dataset,
we found that a value of `groups=8` produces better results
compared to the default value of `groups=32`. Dropout is optional and should be
used where chances of over fitting is high. In the paper, the authors used dropout
only when training on CIFAR10.

In [5]:
from tensorflow.keras.models import load_model

pretrained_encoder = load_model('../saved_models/encoder_cnn_56deg_5var.h5')
pretrained_encoder.summary()

2026-02-02 11:18:48.918387: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-02-02 11:18:48.952218: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-02-02 11:18:48.955484: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Model: "encoder_net"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 32, 64, 5)]       0         
                                                                 
 conv2d (Conv2D)             (None, 32, 64, 32)        672       
                                                                 
 conv2d_1 (Conv2D)           (None, 32, 64, 128)       16512     
                                                                 
 conv2d_2 (Conv2D)           (None, 32, 64, 256)       131328    
                                                                 
 bottleneck (Conv2D)         (None, 32, 64, 512)       524800    
                                                                 
 conv2d_3 (Conv2D)           (None, 32, 64, 256)       524544    
                                                                 
 conv2d_4 (Conv2D)           (None, 32, 64, 128)       

In [6]:
# Extract the first 5 layers
first_five_layers = pretrained_encoder.layers[:5]

# Display the first four layers to confirm
for i, layer in enumerate(first_five_layers):
    print(f"Layer {i}: {layer}")

# Create a new model using these layers
# Get the input of the pre-trained model
input_layer = pretrained_encoder.input

# Get the output of the fourth layer
output_layer = first_five_layers[-1].output

# Create the new model
pretrained_encoder = tf.keras.Model(inputs=input_layer, outputs=output_layer)

# Print the summary of the new model
pretrained_encoder.summary()

Layer 0: <keras.src.engine.input_layer.InputLayer object at 0x760427f2b820>
Layer 1: <keras.src.layers.convolutional.conv2d.Conv2D object at 0x760427f74dc0>
Layer 2: <keras.src.layers.convolutional.conv2d.Conv2D object at 0x760427f750f0>
Layer 3: <keras.src.layers.convolutional.conv2d.Conv2D object at 0x760427f755d0>
Layer 4: <keras.src.layers.convolutional.conv2d.Conv2D object at 0x760427f75ba0>
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 32, 64, 5)]       0         
                                                                 
 conv2d (Conv2D)             (None, 32, 64, 32)        672       
                                                                 
 conv2d_1 (Conv2D)           (None, 32, 64, 128)       16512     
                                                                 
 conv2d_2 (Conv2D)           (None, 32, 64, 256)       13

In [7]:
for layer in pretrained_encoder.layers:
    layer.trainable = False

pretrained_encoder._name = 'encoder'

In [8]:
from layers.denoiser import build_unet_model_c2, build_unet_model_c2_no_cross_attn, build_unet_model_c2_no_encoder, build_unet_model_c2_no_cross_attn_encoder

In [9]:
# Build the unet model
network = build_unet_model_c2(
    img_size_H=img_size_H,
    img_size_W=img_size_W,
    img_channels=img_channels,
    widths=widths,
    has_attention=has_attention,
    num_res_blocks=num_res_blocks,
    norm_groups=norm_groups,
    first_conv_channels=first_conv_channels,
    activation_fn=keras.activations.swish,
    encoder=pretrained_encoder,
)

image_input_past_embed1 shape: (None, 32, 64, 64)
image_input_past_embed2 shape: (None, 32, 64, 64)
image_input_past shape: (None, 32, 64, 64)
image_input_past shape: (None, 2048, 64)
x.shape: (None, 32, 64, 64) temb.shape: (None, 256)


In [10]:
# network.summary()

## Training

We follow the same setup for training the diffusion model as described
in the paper. We use `Adam` optimizer with a learning rate of `2e-4`.
We use `EMA` (Exponential Moving Average) on model parameters with a decay factor of 0.999. We
treat our model as noise prediction network i.e. at every training step, we
input a batch of images and corresponding time steps to our UNet,
and the network outputs the noise as predictions.


**Note: ** We are using mean squared error as the loss function which is aligned with
the paper, and theoretically makes sense. In practice, though, it is also common to
use mean absolute error or Huber loss as the loss function.

In [11]:
class DiffusionModel(keras.Model):
    def __init__(self, network, ema_network, timesteps, gdf_util, ema=0.999):
        super().__init__()
        self.network = network  # denoiser or noise predictor
        self.ema_network = ema_network
        self.timesteps = timesteps
        self.gdf_util = gdf_util
        self.ema = ema

    def train_step(self, data):
        # Unpack the data
        (images, image_input_past1, image_input_past2), y = data
        
        # 1. Get the batch size
        batch_size = tf.shape(images)[0]
        
        # 2. Sample timesteps uniformly
        t = tf.random.uniform(minval=0, maxval=self.timesteps, shape=(batch_size,), dtype=tf.int64)

        with tf.GradientTape() as tape:
            # 3. Sample random noise to be added to the images in the batch
            noise = tf.random.normal(shape=tf.shape(images), dtype=images.dtype)
            print("noise.shape:", noise.shape)
            
            # 4. Diffuse the images with noise
            images_t = self.gdf_util.q_sample(images, t, noise)
            print("images_t.shape:", images_t.shape)
            
            # 5. Pass the diffused images and time steps to the network
            pred_noise = self.network([images_t, t, image_input_past1, image_input_past2], training=True)
            print("pred_noise.shape:", pred_noise.shape)
            
            # 6. Calculate the loss
            loss = self.loss(noise, pred_noise)

        # 7. Get the gradients
        gradients = tape.gradient(loss, self.network.trainable_weights)

        # 8. Update the weights of the network
        self.optimizer.apply_gradients(zip(gradients, self.network.trainable_weights))

        # 9. Updates the weight values for the network with EMA weights
        for weight, ema_weight in zip(self.network.weights, self.ema_network.weights):
            ema_weight.assign(self.ema * ema_weight + (1 - self.ema) * weight)

        # 10. Return loss values
        return {"loss": loss}

    
    def test_step(self, data):
        # Unpack the data
        (images, image_input_past1, image_input_past2), y = data

        # 1. Get the batch size
        batch_size = tf.shape(images)[0]
        
        # 2. Sample timesteps uniformly
        t = tf.random.uniform(minval=0, maxval=self.timesteps, shape=(batch_size,), dtype=tf.int64)

        # 3. Sample random noise to be added to the images in the batch
        noise = tf.random.normal(shape=tf.shape(images), dtype=images.dtype)
        
        # 4. Diffuse the images with noise
        images_t = self.gdf_util.q_sample(images, t, noise)
        
        # 5. Pass the diffused images and time steps to the network
        pred_noise = self.network([images_t, t, image_input_past1, image_input_past2], training=False)
        
        # 6. Calculate the loss
        loss = self.loss(noise, pred_noise)

        # 7. Return loss values
        return {"loss": loss}

### Load trained model

In [12]:
# Build the unet model
network = build_unet_model_c2(
    img_size_H=img_size_H,
    img_size_W=img_size_W,
    img_channels=img_channels,
    widths=widths,
    has_attention=has_attention,
    num_res_blocks=num_res_blocks,
    norm_groups=norm_groups,
    first_conv_channels=first_conv_channels,
    activation_fn=keras.activations.swish,
    encoder=pretrained_encoder,
)

ema_network = build_unet_model_c2(
    img_size_H=img_size_H,
    img_size_W=img_size_W,
    img_channels=img_channels,
    widths=widths,
    has_attention=has_attention,
    num_res_blocks=num_res_blocks,
    norm_groups=norm_groups,
    first_conv_channels=first_conv_channels,
    activation_fn=keras.activations.swish,
    encoder=pretrained_encoder,
)
ema_network.set_weights(network.get_weights())  # Initially the weights are the same

image_input_past_embed1 shape: (None, 32, 64, 64)
image_input_past_embed2 shape: (None, 32, 64, 64)
image_input_past shape: (None, 32, 64, 64)
image_input_past shape: (None, 2048, 64)
x.shape: (None, 32, 64, 64) temb.shape: (None, 256)
image_input_past_embed1 shape: (None, 32, 64, 64)
image_input_past_embed2 shape: (None, 32, 64, 64)
image_input_past shape: (None, 32, 64, 64)
image_input_past shape: (None, 2048, 64)
x.shape: (None, 32, 64, 64) temb.shape: (None, 256)


In [13]:
# Get an instance of the Gaussian Diffusion utilities
gdf_util = GaussianDiffusion(timesteps=total_timesteps)

# Get the model
model = DiffusionModel(
    network=network,
    ema_network=ema_network,
    gdf_util=gdf_util,
    timesteps=total_timesteps,
)

In [14]:
## -------------------- ablation study --------------------
model.load_weights('../checkpoints/ddpm_weather_56c2_56_5var_cp3')  # CoDiCast with 1000 steps
# model.load_weights('../checkpoints/ddpm_weather_56c2_56_5var_cp3_no_cross_attn')
# model.load_weights('../checkpoints/ddpm_weather_56c2_56_5var_cp3_no_encoder')
# model.load_weights('../checkpoints/ddpm_weather_56c2_56_5var_cp3_no_cross_attn_encoder')

## -------------------- diffusion steps --------------------
# model.load_weights('../checkpoints/ddpm_weather_56c2_56_5var_cp3_750')

## Results

In [15]:
from utils.normalization import batch_norm, batch_norm_reverse
from utils.metrics import lat_weighted_rmse_one_var, lat_weighted_acc_one_var

In [16]:
resolution_folder = '56degree'
resolution = '5.625'  
var_num = '5'

test_data_tf = np.load("/mnt/data/sonia/codicast-data/default/concat_2017_2018_" + resolution + "_" + var_num + "var.npy")
test_data_tf = test_data_tf.transpose((0, 2, 3, 1))
test_data_tf.shape

(2920, 32, 64, 5)

In [17]:
test_data_tf_norm = batch_norm(test_data_tf, test_data_tf.shape, batch_size=1460)
test_data_tf_norm.shape

(2920, 32, 64, 5)

In [18]:
test_data_tf_norm_pred = test_data_tf_norm[2:]
test_data_tf_norm_past1 = test_data_tf_norm[:-2]
test_data_tf_norm_past2 = test_data_tf_norm[1:-1]

print(test_data_tf_norm_pred.shape, test_data_tf_norm_past1.shape, test_data_tf_norm_past2.shape)

(2918, 32, 64, 5) (2918, 32, 64, 5) (2918, 32, 64, 5)


### RSME & ACC

In [19]:
from utils.normalization import batch_norm, batch_norm_reverse
from utils.metrics import lat_weighted_rmse_one_var, lat_weighted_acc_one_var

In [20]:
import tensorflow as tf
import numpy as np

def generate_images(model, original_samples, original_samples_past1, original_samples_past2):
    """
    @model: trained denoiser
    @original_samples: it just provides the shape, does not involve generation
    @original_samples_past: conditions from the past
    """
    num_images = original_samples.shape[0]
    img_size_H = original_samples.shape[1]
    img_size_W = original_samples.shape[2]
    img_channels = original_samples.shape[3]
    total_timesteps = model.timesteps  # Ensure this is defined in your model

    # 1. Randomly sample noise (starting point for reverse process)
    samples = tf.random.normal(shape=(num_images, img_size_H, img_size_W, img_channels), dtype=tf.float32)
    
    # 2. Sample from the model iteratively
    for t in reversed(range(0, total_timesteps)):
        tt = tf.cast(tf.fill([num_images], t), dtype=tf.int64)
        pred_noise = model.ema_network.predict([samples, tt, original_samples_past1, original_samples_past2],
                                               verbose=0, 
                                               batch_size=num_images
                                              )
        samples = model.gdf_util.p_sample(pred_noise, samples, tt, clip_denoised=True)
        
    # 3. Return generated samples and original samples
    return original_samples, samples
    # return original_samples.numpy(), samples.numpy()

In [21]:
def predict_autoregressive(model, initial_inputs, prediction_horizon):
    
    predictions = []
    
    original_sample, sample_past1, sample_past2 = initial_inputs[0], initial_inputs[1], initial_inputs[2]  # t, t-2, t-1

    for _ in range(prediction_horizon):
        # Predict the next time step
        original_sample, generated_sample = generate_images(model, original_sample, sample_past1, sample_past2)
        
        print("original_sample.shape:", original_sample.shape, "generated_sample.shape:", generated_sample.shape)
        
        # Append the prediction to the list of predictions
        predictions.append(generated_sample)

        sample_past1 = sample_past2
        sample_past2 = generated_sample
        

    # Concatenate predictions along the time steps axis
    predictions = np.concatenate(predictions, axis=0)
    return predictions

In [22]:
channels = ['geopotential_500', 'temperature_850', 
            '2m_temperature', '10m_u_component_of_wind', '10m_v_component_of_wind']

num_sample = 1  # 2918 for entire test set
num_channel = 5
num_lead = 2


rmse_matrix = np.zeros((num_sample, num_channel, num_lead))
acc_matrix = np.zeros((num_sample, num_channel, num_lead))


for z in range(num_sample):
    #print("sample #", z)
    original_samples = tf.convert_to_tensor(test_data_tf_norm_pred[z:z+num_lead], dtype=tf.float32)
    original_samples_past1 = tf.convert_to_tensor(test_data_tf_norm_past1[z:z+num_lead], dtype=tf.float32)
    original_samples_past2 = tf.convert_to_tensor(test_data_tf_norm_past2[z:z+num_lead], dtype=tf.float32)
    
    print(original_samples.shape, original_samples_past1.shape, original_samples_past2.shape)
    
    initial_inputs = [tf.convert_to_tensor(original_samples[0:1], dtype=tf.float32),
                  tf.convert_to_tensor(original_samples_past1[0:1], dtype=tf.float32), 
                  tf.convert_to_tensor(original_samples_past2[0:1], dtype=tf.float32)
                 ]

    future_predictions = predict_autoregressive(model, initial_inputs, prediction_horizon=num_lead)

    original_samples_unnormlalized = batch_norm_reverse(test_data_tf, test_data_tf.shape, 1459, original_samples)
    generated_samples_unnormlalized = batch_norm_reverse(test_data_tf, test_data_tf.shape, 1459, future_predictions)
    
    print(original_samples_unnormlalized.shape, generated_samples_unnormlalized.shape)
    
    
    for i in range(num_channel):
        print(f'{channels[i]}:')
        for j in range(num_lead):
            # print(f't{num_lead*(j+1)}: {lat_weighted_rmse_one_var(original_samples_unnormlalized[j:j+1], generated_samples_unnormlalized[j:j+1], var_idx=i, resolution=5.625):.5f}')
            rmse_matrix[z][i][j] = lat_weighted_rmse_one_var(original_samples_unnormlalized[j:j+1], generated_samples_unnormlalized[j:j+1], var_idx=i, resolution=5.625)
            acc_matrix[z][i][j] = lat_weighted_acc_one_var(original_samples_unnormlalized[j:j+1], generated_samples_unnormlalized[j:j+1], var_idx=i, resolution=5.625, clim=test_data_tf)
        print('\n')

    # print("rmse_matrix.shape:", rmse_matrix.shape)
    # print("acc_matrix.shape:", acc_matrix.shape)

(2, 32, 64, 5) (2, 32, 64, 5) (2, 32, 64, 5)


2026-02-02 11:20:02.883937: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


original_sample.shape: (1, 32, 64, 5) generated_sample.shape: (1, 32, 64, 5)
original_sample.shape: (1, 32, 64, 5) generated_sample.shape: (1, 32, 64, 5)
(2, 32, 64, 5) (2, 32, 64, 5)
geopotential_500:


temperature_850:


2m_temperature:


10m_u_component_of_wind:


10m_v_component_of_wind:




In [23]:
# ddpm_weather_56c2_56_5var_cp3_1000
rmse_matrix_mean = np.mean(rmse_matrix, axis=0)
rmse_matrix_mean

array([[163.0032586 , 229.52591455],
       [  1.38950849,   1.81878377],
       [  0.7193365 ,   1.19036351],
       [  0.92970184,   1.50177808],
       [  0.94680416,   1.4540947 ]])

In [24]:
# ddpm_weather_56c2_56_5var_cp3_1000
acc_matrix_mean = np.mean(acc_matrix, axis=0)
acc_matrix_mean

array([[0.99599583, 0.988768  ],
       [0.9763641 , 0.95890903],
       [0.997446  , 0.99269987],
       [0.96800862, 0.92115102],
       [0.97244884, 0.93607362]])

In [25]:
# ddpm_weather_56c2_56_5var_cp3_2000
rmse_matrix_mean = np.mean(rmse_matrix, axis=0)
rmse_matrix_mean

array([[163.0032586 , 229.52591455],
       [  1.38950849,   1.81878377],
       [  0.7193365 ,   1.19036351],
       [  0.92970184,   1.50177808],
       [  0.94680416,   1.4540947 ]])

In [26]:
# ddpm_weather_56c2_56_5var_cp3_1500
rmse_matrix_mean = np.mean(rmse_matrix, axis=0)
rmse_matrix_mean

array([[163.0032586 , 229.52591455],
       [  1.38950849,   1.81878377],
       [  0.7193365 ,   1.19036351],
       [  0.92970184,   1.50177808],
       [  0.94680416,   1.4540947 ]])

In [27]:
# ddpm_weather_56c2_56_5var_cp3_750
rmse_matrix_mean = np.mean(rmse_matrix, axis=0)
rmse_matrix_mean

array([[163.0032586 , 229.52591455],
       [  1.38950849,   1.81878377],
       [  0.7193365 ,   1.19036351],
       [  0.92970184,   1.50177808],
       [  0.94680416,   1.4540947 ]])

In [28]:
# ddpm_weather_56c2_56_5var_cp3_500
rmse_matrix_mean = np.mean(rmse_matrix, axis=0)
rmse_matrix_mean

array([[163.0032586 , 229.52591455],
       [  1.38950849,   1.81878377],
       [  0.7193365 ,   1.19036351],
       [  0.92970184,   1.50177808],
       [  0.94680416,   1.4540947 ]])

In [29]:
# ddpm_weather_56c2_56_5var_cp3_250
rmse_matrix_mean = np.mean(rmse_matrix, axis=0)
rmse_matrix_mean

array([[163.0032586 , 229.52591455],
       [  1.38950849,   1.81878377],
       [  0.7193365 ,   1.19036351],
       [  0.92970184,   1.50177808],
       [  0.94680416,   1.4540947 ]])